# Ensemble starter — text feature bagging, bootstrap bagging and spatial corrections

**Target:** aim for **sub-7% off-market MdAPE** on full-strength chronological evaluation. Our current best observed full-history MdAPE is **7.4857%**; sub-7 would require roughly a 6.5% relative reduction. This is a stretch goal, not a prediction.

This notebook starts the ensemble registry and a small **optional** feature/row-bagged XGBoost experiment. It uses the 47 selected binary-BoW terms and the training-fitted ~1,000 candidate vocabulary, instead of limiting the ensemble to Tensor GAM and Matérn corrections.

See [ENSEMBLE_CANDIDATES.md](../ENSEMBLE_CANDIDATES.md) for the full roadmap. Keep off-market and list-aware models separate, and do not compare pilot raw scores to the old full-history benchmark.

**Storage:** new fitted snapshots live in `saved_models/off_market/<name>/` or
`saved_models/list_aware/<name>/`, while old root-level snapshots remain
readable as legacy. A bagging run saves **one shared fitted feature transformer
plus each individually addressable tree member** using `save_model(..., track="off_market")`.
Saving is enabled for the opt-in pilot; nothing is saved until training is enabled.

**Design:** train chronological feature providers on ALL eligible historical sales; then select binary-BoW columns and optionally bootstrap XGBoost fitting rows. Historical comparable-sale features are **not** regenerated from bootstrapped sales. Validation properties are never used to choose the vocabulary or training rows.


In [1]:
# Locate the checkout from Jupyter/VS Code, including nested notebook folders.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("DATA_DASH_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "data.py").is_file() and (p / "models").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DATA_DASH_ROOT to your Data Dash checkout before running this notebook.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display
import pandas as pd
from models.ensemble_bagging import starter_specs, SELECTED_TERMS

ENSEMBLE_SEED = 42
ALL_BAG_SPECS = starter_specs(ENSEMBLE_SEED)
print("Selected reference BoW terms:", len(SELECTED_TERMS))
print("Candidate member configurations:", len(ALL_BAG_SPECS))
display(pd.DataFrame([s.record() for s in ALL_BAG_SPECS]))

MODEL_REGISTRY = pd.DataFrame([
    ("xgb_selected_47", "binary_bow", "Reference 47 selected terms"),
    ("xgb_full_binary", "binary_bow", "Training-fitted broad vocabulary"),
    ("xgb_selected_plus_random", "feature_bag", "Keep selected + random additions"),
    ("xgb_weighted_text_bags", "feature_bag", "4x selected-term sampling prior"),
    ("xgb_row_bootstraps", "row_bag", "Same features, bootstrapped fitting rows"),
    ("xgb_feature_row_joint", "joint_bag", "Weighted term sampling + bootstrap"),
    ("xgb_splines_geo", "spatial_features", "Tuned XGBoost with splines + geography"),
    ("xgb_tensor_gam", "residual", "GAM correction on true earlier OOT errors"),
    ("xgb_spatial_matern", "residual", "Tuned spatial GP correction"),
    ("xgb_spacetime_matern", "residual", "Tuned spatiotemporal GP correction"),
    ("lgb_catboost", "other_models", "Future independently tuned challenger"),
], columns=["model_id", "diversity_source", "purpose"])
display(MODEL_REGISTRY)


Selected reference BoW terms: 47
Candidate member configurations: 11


,name,text_mode,n_text,selected_weight,row_mode,row_fraction,seed
0,selected_47_reference,selected,150,4.0,all,1.0,42
1,full_vocab_binary,full,150,4.0,all,1.0,42
2,selected_plus_100,selected_plus,100,4.0,all,1.0,43
3,uniform_150_a,uniform,150,4.0,all,1.0,44
4,weighted_150_a,weighted,150,4.0,all,1.0,45
5,weighted_150_b,weighted,150,4.0,all,1.0,46
6,weighted_150_bootstrap_a,weighted,150,4.0,bootstrap,1.0,47
7,weighted_150_bootstrap_b,weighted,150,4.0,bootstrap,1.0,48
8,selected_47_bootstrap_a,selected,150,4.0,bootstrap,1.0,49
9,selected_47_bootstrap_b,selected,150,4.0,bootstrap,1.0,50


,model_id,diversity_source,purpose
0,xgb_selected_47,binary_bow,Reference 47 selected terms
1,xgb_full_binary,binary_bow,Training-fitted broad vocabulary
2,xgb_selected_plus_random,feature_bag,Keep selected + random additions
3,xgb_weighted_text_bags,feature_bag,4x selected-term sampling prior
4,xgb_row_bootstraps,row_bag,"Same features, bootstrapped fitting rows"
5,xgb_feature_row_joint,joint_bag,Weighted term sampling + bootstrap
6,xgb_splines_geo,spatial_features,Tuned XGBoost with splines + geography
7,xgb_tensor_gam,residual,GAM correction on true earlier OOT errors
8,xgb_spatial_matern,residual,Tuned spatial GP correction
9,xgb_spacetime_matern,residual,Tuned spatiotemporal GP correction


## 1 · Train a small text / row-bagging pilot (opt-in)

We start with August 2025 rather than the repeatedly analyzed Sep–Nov months. This is **development screening**, not an untouched final test.

The pilot keeps the existing tuned XGBoost parameter family, uses 250 trees and at most 6,000 historical sales, and compares 6 complementary variations. Use the same validation homes and dates for every model. If you add or remove members, **change the output directory** to avoid mixing cached results.

The first run may be slower while the full housing feature matrix and historical comparables are built; only the estimator fitting rows and binary text columns vary across bag members.


In [2]:
from models.ensemble_feature_bagging_pilot import run_feature_bagging_pilot

RUN_ENSEMBLE_BAG_PILOT = False  # Change to True to train the first candidates
SAVE_FITTED_ENSEMBLE_MODELS = True  # Save the full reusable bundle when training
BAG_PILOT_SPECS = (
    ALL_BAG_SPECS[0],  # 47 selected terms / all eligible training rows
    ALL_BAG_SPECS[1],  # full 1000-ish binary vocabulary
    ALL_BAG_SPECS[2],  # selected 47 plus random 100
    ALL_BAG_SPECS[3],  # uniform 150-term bag
    ALL_BAG_SPECS[4],  # weighted 150-term bag
    ALL_BAG_SPECS[6],  # weighted 150-term bag + row bootstrap
    ALL_BAG_SPECS[8],  # selected 47 + row bootstrap
)

BAG_PILOT_OUTPUT = "outputs/ensemble_feature_bagging_smoke"
if RUN_ENSEMBLE_BAG_PILOT:
    bag_scores, bag_predictions, bag_manifest = run_feature_bagging_pilot(
        validation_month="2025-08",
        train_start="2022-01-01",
        train_cap=6_000,
        validation_cap=350,
        trees=250,
        specs=BAG_PILOT_SPECS,
        seed=ENSEMBLE_SEED,
        max_extra_terms=1_000,
        min_df=10,
        output_dir=BAG_PILOT_OUTPUT,
        resume=True,
        save_fitted_models=SAVE_FITTED_ENSEMBLE_MODELS,
    )
    display(bag_scores.round(4))
    display(bag_manifest[[
        "name", "text_mode", "n_terms", "row_mode",
        "n_training_rows", "unique_training_rows"
    ]])
    print("Predictions and exact selected terms are saved in:", BAG_PILOT_OUTPUT)
    if SAVE_FITTED_ENSEMBLE_MODELS:
        print("Fitted ensemble bundle is stored under saved_models/off_market/")


In [3]:
# Simple PREDECLARED log-price blends; no tuning against the scoring homes.
# This is exploratory—don't promote based solely on the best August result.
from pathlib import Path
from models.ensemble_blending import score_blends
from models.ensemble_bagging import starter_specs

pilot_path = Path(BAG_PILOT_OUTPUT) / "predictions.csv"
if pilot_path.is_file():
    bag_predictions = pd.read_csv(pilot_path, dtype={"ml_num": str})
    fixed_blends = {
        "equal_reference_weighted": {
            "selected_47_reference": 0.5, "weighted_150_a": 0.5
        },
        "equal_reference_plus100": {
            "selected_47_reference": 0.5, "selected_plus_100": 0.5
        },
        "reference_weighted_bootstrap": {
            "selected_47_reference": 0.5,
            "weighted_150_bootstrap_a": 0.5
        },
        "equal_3_member_bag": {
            "selected_47_reference": 1,
            "selected_plus_100": 1,
            "weighted_150_bootstrap_a": 1,
        },
    }
    available = set(bag_predictions.member)
    fixed_blends = {
        k: v for k, v in fixed_blends.items() if set(v).issubset(available)
    }
    if fixed_blends:
        blend_scores, blend_predictions = score_blends(
            bag_predictions, fixed_blends
        )
        display(blend_scores.round(4))
        blend_predictions.to_csv(
            Path(BAG_PILOT_OUTPUT) / "fixed_blend_predictions.csv",
            index=False,
        )
        # Error correlation identifies duplication, not necessarily diversity gains.
        wide = bag_predictions.pivot(
            index=["month", "ml_num"], columns="member",
            values="predicted_price"
        )
        actual = bag_predictions.groupby(["month", "ml_num"]).sold_price.first()
        import numpy as np
        log_error = pd.DataFrame(
            {col: np.log(actual.loc[wide.index].to_numpy()) -
                  np.log(wide[col].to_numpy())
             for col in wide.columns},
            index=wide.index,
        )
        display(log_error.corr().round(3))
else:
    print("Run the opt-in bagging pilot first; no training occurs in this cell.")


Run the opt-in bagging pilot first; no training occurs in this cell.


## 2 · Separate diagnostic: are the spatial residual corrections complementary?

This block reads previously saved Matérn confirmation predictions and shows only **fixed 50/50 log-price blends** with Tensor GAM. No new GP/XGBoost fit is required.

**Important:** September–November 2025 have already been examined repeatedly, so this is an exploratory complementarity check, *not a legitimate untouched ensemble-selection result*. Do not tune blend weights on these predictions.


In [4]:
from pathlib import Path
from models.ensemble_blending import score_blends
from IPython.display import display

spatial_file = Path(
    "outputs/spacetime_matern_chronological_tuning/confirmation_predictions.csv"
)
if spatial_file.is_file():
    spatial_predictions = pd.read_csv(spatial_file, dtype={"ml_num": str})
    spatial_blends = {
        "gam_spatial_50_50": {
            "xgb_tensor_gam": .5, "xgb_spatial_matern_tuned": .5
        },
        "gam_spacetime_50_50": {
            "xgb_tensor_gam": .5, "xgb_spacetime_matern_tuned": .5
        },
        "gam_spatial_spacetime_equal": {
            "xgb_tensor_gam": 1,
            "xgb_spatial_matern_tuned": 1,
            "xgb_spacetime_matern_tuned": 1,
        },
    }
    spatial_scores, spatial_new_preds = score_blends(
        spatial_predictions, spatial_blends, variant_col="variant"
    )
    singles = spatial_predictions.groupby("variant").apply(
        lambda g: pd.Series({
            "homes": len(g),
            "mdape": (100*abs(g.predicted_price-g.sold_price)/g.sold_price).median(),
            "mape": (100*abs(g.predicted_price-g.sold_price)/g.sold_price).mean(),
        }),
        include_groups=False,
    ).reset_index().rename(columns={"variant": "member"})
    display(pd.concat([singles, spatial_scores], ignore_index=True)
            .sort_values(["mdape", "mape"]).round(4))
else:
    print("No local Matérn confirmation CSV yet; this section is optional.")


,member,homes,mdape,mape
4,gam_spacetime_50_50,2700.0,10.6762,13.4945
5,gam_spatial_spacetime_equal,2700.0,10.7030,13.4398
1,xgb_spacetime_matern_tuned,2700.0,10.7129,13.6247
3,xgb_tensor_gam,2700.0,10.7983,13.4371
0,baseline,2700.0,10.8761,13.5692
6,gam_spatial_50_50,2700.0,10.8780,13.3906
2,xgb_spatial_matern_tuned,2700.0,10.9641,13.3782


## 3 · Does blending add information, or only shrink corrections?

**Prediction-only, no retraining.** Read the Matérn chronological confirmation CSV, which contains the identical September–November homes under `baseline`, `xgb_tensor_gam` and both tuned Matérn corrections.

All models share the same XGBoost baseline. Recover each model's **effective** log correction as

\[
r_G=\log(P_{GAM})-\log(P_{XGB}),\qquad
r_M=\log(P_{Matérn})-\log(P_{XGB}).
\]

These are already **clipped and scaled** by their respective earlier fitted correction strength. This experiment does *not* refit GAM, GP, XGBoost or blending weights.

Compare:
- GAM and each Matérn alone;
- `0.5*r_G` and `0.5*r_M` (individual shrinkage controls);
- `0.5*r_G + 0.5*r_M` (the existing 50/50 geometric blend);
- `0.25*r_G + 0.25*r_M` (a doubly-shrunk blend control).

The important paired test is whether the blend beats **both individually shrunk** corrections on the *same homes*, in MdAPE and MAPE. Results include month-specific scores, month-stratified paired bootstrap intervals, per-home improvements, and correlation of the effective log corrections.

**Research caveat:** Sep–Nov 2025 has already been examined repeatedly; this is a retrospective diagnostic, *not independent evidence* to promote weights into the final ensemble. Do not pick or optimize weights from these scores.

In [5]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from models.ensemble_shrinkage_diagnostic import run_shrinkage_vs_blending

RUN_SHRINKAGE_VS_BLEND = True  # Only reads the existing confirmation predictions
SHRINKAGE_SOURCE = Path(
    "outputs/spacetime_matern_chronological_tuning/confirmation_predictions.csv"
)
SHRINKAGE_OUTPUT = "outputs/ensemble_shrinkage_vs_spatial_blending"

if RUN_SHRINKAGE_VS_BLEND:
    if not SHRINKAGE_SOURCE.is_file():
        print(
            "Missing Matérn confirmation predictions:", SHRINKAGE_SOURCE,
            "\nFirst run the frozen Matérn confirmation experiment."
        )
    else:
        original_predictions = pd.read_csv(
            SHRINKAGE_SOURCE, dtype={"ml_num": str}
        )
        (shrink_pooled, shrink_monthly, shrink_pairs,
         shrink_correlations, shrink_predictions) = run_shrinkage_vs_blending(
            original_predictions,
            shrinkage=0.5,
            bootstrap_repeats=500,
            seed=42,
            output_dir=SHRINKAGE_OUTPUT,
        )
        print("All fixed correction strategies (lower is better):")
        display(shrink_pooled.round(4))
        print("Most revealing comparison: fixed 50/50 vs controls")
        display(shrink_pairs[[
            "challenger", "reference", "delta_mdape_pp",
            "mdape_ci_low", "mdape_ci_high",
            "delta_mape_pp", "mape_ci_low", "mape_ci_high",
            "homes_improved_pct", "months_better_mdape",
        ]].round(4))
        print("Monthly MdAPE / MAPE:")
        display(shrink_monthly[[
            "month", "variant", "homes", "mdape", "mape"
        ]].round(4))
        print("Effective correction-pattern correlations:")
        display(shrink_correlations.round(4))
        print("Saved predictions and comparisons:", SHRINKAGE_OUTPUT)

All fixed correction strategies (lower is better):


,variant,homes,mdape,mape,p90_ape,median_signed_pct,delta_mdape_vs_baseline_pp,delta_mape_vs_baseline_pp
0,gam_spacetime_50_50,2700,10.6762,13.4945,28.1022,1.5890,-0.1999,-0.0747
1,spacetime_half_strength,2700,10.6941,13.5589,28.2355,1.9253,-0.1820,-0.0103
2,xgb_spacetime_matern_tuned,2700,10.7129,13.6247,28.4352,1.9962,-0.1632,0.0556
3,spatial_half_strength,2700,10.7917,13.4453,28.0079,1.3929,-0.0844,-0.1239
4,xgb_tensor_gam,2700,10.7983,13.4371,27.8634,1.3129,-0.0778,-0.1321
5,gam_spatial_blend_half_strength,2700,10.8054,13.4649,27.9434,1.5927,-0.0707,-0.1042
6,gam_spacetime_blend_half_strength,2700,10.8479,13.5200,28.0728,1.7915,-0.0282,-0.0491
7,gam_half_strength,2700,10.8706,13.4945,28.0126,1.7700,-0.0055,-0.0747
8,baseline,2700,10.8761,13.5692,28.0119,2.0450,0.0000,0.0000
9,gam_spatial_50_50,2700,10.8780,13.3906,27.7169,0.8738,0.0019,-0.1786


Most revealing comparison: fixed 50/50 vs controls


,challenger,reference,delta_mdape_pp,mdape_ci_low,mdape_ci_high,delta_mape_pp,mape_ci_low,mape_ci_high,homes_improved_pct,months_better_mdape
0,gam_spacetime_50_50,baseline,-0.1999,-0.4033,-0.0057,-0.0747,-0.1262,-0.0245,51.3704,3
1,gam_spacetime_50_50,xgb_tensor_gam,-0.1221,-0.3187,0.0451,0.0574,0.0080,0.1052,49.2222,2
2,gam_spacetime_50_50,xgb_spacetime_matern_tuned,-0.0368,-0.2747,0.0923,-0.1302,-0.1792,-0.0851,53.2593,2
3,gam_spacetime_50_50,gam_half_strength,-0.1944,-0.3805,-0.0274,-0.0000,-0.0440,0.0433,50.2963,3
4,gam_spacetime_50_50,spacetime_half_strength,-0.0179,-0.1424,0.0969,-0.0644,-0.0874,-0.0422,50.9630,1
5,gam_spacetime_50_50,gam_spacetime_blend_half_strength,-0.1717,-0.2732,0.0575,-0.0255,-0.0523,-0.0003,50.8148,2
6,gam_spatial_50_50,baseline,0.0019,-0.1946,0.1787,-0.1786,-0.2289,-0.1250,51.7037,2
7,gam_spatial_50_50,xgb_tensor_gam,0.0797,-0.1281,0.1922,-0.0465,-0.0765,-0.0178,50.7407,1
8,gam_spatial_50_50,xgb_spatial_matern_tuned,-0.0861,-0.2306,0.0663,0.0124,-0.0156,0.0402,50.9259,3
9,gam_spatial_50_50,gam_half_strength,0.0074,-0.1981,0.1376,-0.1039,-0.1414,-0.0696,49.7778,1


Monthly MdAPE / MAPE:


,month,variant,homes,mdape,mape
12,2025-09,spacetime_half_strength,900,10.1506,13.4192
15,2025-09,gam_spacetime_50_50,900,10.2800,13.3172
9,2025-09,xgb_spacetime_matern_tuned,900,10.3074,13.5094
18,2025-09,gam_spacetime_blend_half_strength,900,10.3758,13.3499
0,2025-09,baseline,900,10.3827,13.4105
30,2025-09,gam_spatial_blend_half_strength,900,10.4349,13.3148
24,2025-09,spatial_half_strength,900,10.4778,13.3450
3,2025-09,xgb_tensor_gam,900,10.5921,13.2047
6,2025-09,gam_half_strength,900,10.6240,13.2934
27,2025-09,gam_spatial_50_50,900,10.6887,13.2584


Effective correction-pattern correlations:


,matern,homes,gam_matern_log_correction_pearson,gam_matern_log_correction_spearman,sign_agreement_pct,gam_median_abs_correction,matern_median_abs_correction,gam_p95_abs_correction,matern_p95_abs_correction
0,xgb_spacetime_matern_tuned,2700,0.1679,0.1853,58.8148,0.0076,0.0107,0.0223,0.0545
1,xgb_spatial_matern_tuned,2700,0.4516,0.4083,70.4444,0.0076,0.0126,0.0223,0.0370


Saved predictions and comparisons: outputs/ensemble_shrinkage_vs_spatial_blending


## Next checkpoint: iterative ensemble factory

The new **shrinkage-versus-blending** test helps determine whether
GAM + Matérn is genuinely complementary or simply a lower-strength GP/GAM
correction. Keep both its per-home predictions and paired comparisons in the
ensemble candidate registry. After that, the most valuable new signal is whether **weighted term bags + bootstrap samples** generate good predictions with meaningfully different errors from the 47-feature reference. Select promising members using *earlier out-of-time folds*; then promote a small, frozen ensemble to larger 700-tree/full-history training.

Keep model roster, training cutoffs, feature bag terms, bootstrap seeds, per-home predictions and fixed or OOT-fitted blend weights together. Without these, apparent ensemble gains are hard to reproduce. For a production-facing score below 7%, confirm on a genuinely separate chronological period if available; the old Dec 2025–May 2026 six-month benchmark has already been inspected.


## 3 · Load a fitted ensemble bundle (without retraining)

The model snapshot includes the already-fitted feature provider, its training-only text vocabulary, the selected per-member feature columns, fitted XGBoost estimators, and the log-target offset. Track-specific storage lets similarly named off-market and list-aware bundles coexist. The original dual-track `saved_models/XGBoost_text_binary` remains legacy and unchanged.

You can load the saved artifact with `load_model(name, track="off_market")`. Use `predict_member(hidden_homes, member_name)` for an individual ensemble member, `predict_members(hidden_homes)` to obtain all member predictions, or `predict(hidden_homes, weights=...)` for a fixed-weight geometric blend.


In [6]:
from models.save_model import list_saved_models, list_model_catalog, load_model
print("Saved off-market snapshots:", list_saved_models(track="off_market"))
print("Saved list-aware snapshots:", list_saved_models(track="list_aware"))
display(pd.DataFrame(list_model_catalog()))

# Enable only after the pilot has saved at least one fitted ensemble bundle.
LOAD_SAVED_ENSEMBLE = False
SAVED_ENSEMBLE_NAME = ""  # Use the name printed by the save step.
if LOAD_SAVED_ENSEMBLE:
    if not SAVED_ENSEMBLE_NAME:
        raise ValueError("Set SAVED_ENSEMBLE_NAME to an off-market bundle name")
    saved_ensemble = load_model(
        SAVED_ENSEMBLE_NAME, track="off_market"
    )
    print("Available fitted members:", sorted(saved_ensemble.members))
    print("Example: saved_ensemble.predict_member(hidden_homes, 'weighted_150_a')")


Saved off-market snapshots: []
Saved list-aware snapshots: []


,track,name,relative_path
0,legacy,CatBoost_Optuna_v1,CatBoost_Optuna_v1/
1,legacy,LightGBM,LightGBM/
2,legacy,LightGBM_Optuna_v1,LightGBM_Optuna_v1/
3,legacy,LightGBM_text_binary,LightGBM_text_binary/
4,legacy,XGBoost,XGBoost/
5,legacy,XGBoost_Optuna_v1,XGBoost_Optuna_v1/
6,legacy,XGBoost_text_binary,XGBoost_text_binary/
7,legacy,catboost_spatiotemporal_900,catboost_spatiotemporal_900/


---

## 4 · Iterative AutoML ensemble factory (first working implementation)

**Pipeline:** randomized 47-prioritized binary-BoW feature bags → independent bootstrap/subbag training rows → randomized legal space-time feature add-ons → cheap diagnostic screen → bounded **Optuna** tuning on early months → ensemble-weight admission on later matched homes → a **separately enabled, one-time** shadow-audit report after selection.

Only **XGBoost** is trainable in version 1; the ensemble-selection layer is separate so we can add CatBoost, LightGBM, and model-specific Tensor GAM / Matérn corrections once this foundation is validated.

Default cohorts:
- **March–April 2025:** Optuna training objective, preceding-month historical fitting.
- **May–June 2025:** candidate acceptance and fixed nonnegative log-price blend weights.
- **July–August 2025:** audit predictions, not consulted for parameter choice or acceptance.

These dates have been involved in earlier research. The last segment is shielded from **this factory's** decisions, **not a newly untouched competition holdout**. The repeatedly inspected December 2025–May 2026 six-month benchmark is also not a new holdout.

**Admission criterion:** a candidate must improve the current ensemble's `0.65 × MdAPE + 0.35 × MAPE` objective by at least **0.02 percentage points**, with MAPE deterioration no larger than **0.03 points**. Trial blends use a small, predefined weight grid. This still reuses the acceptance months across iterations, so keep the first run extremely small and treat the audit gap as a warning sign rather than a green light for unlimited tuning.

**Audit guard:** Leave `RUN_AUTOML_SHADOW_AUDIT=False` while developing.
After the roster is frozen, enable it once to inspect the audit. The
factory will then refuse to add new generations to that output directory,
preventing feedback from the revealed audit results.

**Stop and resume:** You can press **Interrupt Kernel / Ctrl+C**, close the notebook, or restart your machine. Re-running the SAME factory cell, with the same `output_dir` and settings, automatically restores the committed candidate roster, prior Optuna trials, previously completed tuning folds and cached held-out predictions. A model fit stopped in mid-fit is restarted; boosting rounds within that fit are **not** checkpointed. The factory does not continue training while the notebook is closed.

New files: `progress.json` reports the current stage; `roster.json` is the authoritative checkpoint, written atomically. A persistent `.run.lock` protects against accidentally running two factory processes in the same directory; its OS lock releases when the process dies, so **do not delete it manually**. You can safely increase `AUTOML_GENERATIONS` to extend the same protocol **until you open the shadow audit**. Other settings must remain fixed.

**Saving:** Optuna studies, candidate forecasts, member roster, current weights and shadow-audit predictions persist in `outputs/ensemble_automl_factory_v1/`. Optionally, accepted fitted model bundles (including transformers and individual tree models) are saved in `saved_models/off_market/`; list-aware ensembles will be trained separately.

**The eventual full ensemble factory** will add independent model families, candidate replacement, truly separate chronological outer tests and full-strength 700-tree refits. This run is an exploratory engineering pilot, **not** a full final ensemble optimizer.


In [9]:
# v1 factory run: safe, small default budget; explicit opt-in for training.
# Optional dependency, install once in the notebook terminal: pip install optuna
from models.ensemble_automl_factory import run_automl_factory, run_automl_shadow_audit
from IPython.display import display

RUN_AUTOML_FACTORY = True
AUTOML_GENERATIONS = 1
AUTOML_CANDIDATES_PER_GENERATION = 2
AUTOML_OPTUNA_TRIALS = 2
AUTOML_SAVE_ACCEPTED_MODELS = True

AUTOML_SETTINGS = dict(
        tuning_months=("2025-03", "2025-04"),
        selection_months=("2025-05", "2025-06"),
        audit_months=("2025-07", "2025-08"),
        train_start="2022-01-01",
        train_cap=5_000,
        validation_cap=250,
        trees=220,
        max_extra_terms=1_000,
        min_df=10,
        candidates_per_generation=AUTOML_CANDIDATES_PER_GENERATION,
        generations=AUTOML_GENERATIONS,
        optuna_trials=AUTOML_OPTUNA_TRIALS,
        minimum_ensemble_gain_pp=0.02,
        max_mape_regression_pp=0.03,
        max_members=8,
        seed=42,
        screen_max_mdape=22.0,
        save_accepted_snapshots=AUTOML_SAVE_ACCEPTED_MODELS,
        output_dir="outputs/ensemble_automl_factory_v1",
)
if RUN_AUTOML_FACTORY:
    automl_summary, automl_events, automl_selection_predictions = run_automl_factory(
        **AUTOML_SETTINGS
    )
    print("Final selected members / effective log-price weights:")
    display(pd.Series(automl_summary["effective_member_weights"]))
    display(pd.DataFrame([
        {"cohort": "selection", **automl_summary["selection_metrics"]},
    ]).round(4))
    display(automl_events)
    print("Cumulative run artifacts: outputs/ensemble_automl_factory_v1/")
    print("Audit still sealed. Open it ONCE in the separate cell below.")



[1/1] Trying selected_47_splines_geo_reference: {'name': 'selected_47_splines_geo_reference', 'seed': 42, 'text_mode': 'selected', 'n_text': 150, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ('spacetime_splines', 'geo_context')}


[I 2026-10-22 21:55:26,542] A new study created in RDB with name: factory_v1_selected_47_splines_geo_reference
[I 2026-10-22 21:55:40,720] Trial 0 finished with value: 11.920319699784642 and parameters: {'max_depth': 8, 'min_child_weight': 16.684619386293466, 'learning_rate': 0.09522271657253033, 'reg_lambda': 19.50331236061207, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713}. Best is trial 0 with value: 11.920319699784642.
[I 2026-10-22 21:55:43,962] Trial 1 finished with value: 11.765513964843933 and parameters: {'max_depth': 4, 'min_child_weight': 13.588238747930372, 'learning_rate': 0.045007648899325604, 'reg_lambda': 11.688766334086836, 'subsample': 0.9975275150067292, 'colsample_bytree': 0.739176414777492}. Best is trial 1 with value: 11.765513964843933.


  Saved fitted member: saved_models/off_market/factory_v1_selected_47_splines_geo_reference
  ADMIT gain=None ensemble size=1

[1/1] Trying g000_c001: {'name': 'g000_c001', 'seed': 942484272, 'text_mode': 'selected', 'n_text': 150, 'selected_weight': 4.0, 'row_mode': 'subbag', 'row_fraction': 0.8, 'addons': ('spacetime_splines', 'geo_context')}


[I 2026-10-22 21:56:03,495] A new study created in RDB with name: factory_v1_g000_c001
[I 2026-10-22 21:56:17,856] Trial 0 finished with value: 12.641375781425529 and parameters: {'max_depth': 8, 'min_child_weight': 16.684619386293466, 'learning_rate': 0.09522271657253033, 'reg_lambda': 19.50331236061207, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713}. Best is trial 0 with value: 12.641375781425529.
[I 2026-10-22 21:56:28,704] Trial 1 finished with value: 12.577902070232023 and parameters: {'max_depth': 9, 'min_child_weight': 14.855442238268669, 'learning_rate': 0.1148036327586377, 'reg_lambda': 27.77029246913031, 'subsample': 0.9765805822930845, 'colsample_bytree': 0.8656608166592223}. Best is trial 1 with value: 12.577902070232023.


  Saved fitted member: saved_models/off_market/factory_v1_g000_c001
  ADMIT gain=0.07841187789221138 ensemble size=2

AutoML ensemble selection: {'homes': 500, 'mdape': 11.068321311888534, 'mape': 13.911057315580218, 'p90_ape': 30.380027612683197}
Shadow audit NOT opened; explicitly enable its separate cell.
Final selected members / effective log-price weights:


selected_47_splines_geo_reference    0.9
g000_c001                            0.1
dtype: float64

,cohort,homes,mdape,mape,p90_ape
0,selection,500,11.0683,13.9111,30.38


,generation,candidate,params,tuning_score,screen,admitted,weight,marginal_gain,solo,error_corr_with_ensemble,updated_metrics,elapsed_seconds
0,0,"{'name': 'selected_47_splines_geo_reference', ...","{'n_estimators': 700, 'learning_rate': 0.04500...",11.765514,"{'homes': 250, 'mdape': 10.594628057644677, 'm...",True,1.0,NaN,"{'homes': 500, 'mdape': 11.14616466396933, 'ma...",NaN,"{'homes': 500, 'mdape': 11.14616466396933, 'ma...",33.785465
1,0,"{'name': 'g000_c001', 'seed': 942484272, 'text...","{'n_estimators': 700, 'learning_rate': 0.11480...",12.577902,"{'homes': 250, 'mdape': 10.385089034536826, 'm...",True,0.1,0.078412,"{'homes': 500, 'mdape': 11.686882044065246, 'm...",0.890462,"{'homes': 500, 'mdape': 11.068321311888534, 'm...",49.453188


Cumulative run artifacts: outputs/ensemble_automl_factory_v1/
Audit still sealed. Open it ONCE in the separate cell below.


In [11]:
# One-time audit. DO NOT enable while you are still adding generations.
# The factory locks the roster after opening this diagnostic.
RUN_AUTOML_SHADOW_AUDIT = True

if RUN_AUTOML_SHADOW_AUDIT:
    frozen_summary, shadow_audit_predictions = run_automl_shadow_audit(
        **AUTOML_SETTINGS
    )
    print("Selection versus shadow audit:")
    display(pd.DataFrame([
        {"cohort": "selection", **frozen_summary["selection_metrics"]},
        {"cohort": "shadow audit", **frozen_summary["shadow_audit_metrics"]},
        {"cohort": "audit first reference",
         **frozen_summary["audit_first_reference_metrics"]},
    ]).round(4))
    print("Audit sealed to frozen roster at:",
          "outputs/ensemble_automl_factory_v1/shadow_audit_lock.json")


Selection versus shadow audit:


,cohort,homes,mdape,mape,p90_ape
0,selection,500,11.0683,13.9111,30.3800
1,shadow audit,500,11.2745,14.8734,32.0059
2,audit first reference,500,11.2129,14.9921,32.2307


Audit sealed to frozen roster at: outputs/ensemble_automl_factory_v1/shadow_audit_lock.json


In [12]:
# Inspect a completed or interrupted study without retraining anything.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

factory_dir = Path("outputs/ensemble_automl_factory_v1")
for filename in ("progress.json", "roster.json", "summary.json", "events.json"):
    path = factory_dir / filename
    if path.is_file():
        print(filename)
        payload = json.loads(path.read_text(encoding="utf-8"))
        if filename == "progress.json":
            display(pd.Series(payload, name="Last durable stage"))
        elif filename == "events.json":
            display(pd.DataFrame(payload))
        elif filename == "roster.json":
            print("Accepted members:", [
                m["candidate"]["name"] for m in payload["members"]
            ])
            print("Processed candidate count:", len(payload["processed"]))
        else:
            display(pd.Series(payload["effective_member_weights"]))
            print("Selection:", payload["selection_metrics"])
            if "shadow_audit_metrics" in payload:
                print("Shadow audit:", payload["shadow_audit_metrics"])
            else:
                print("Shadow audit has NOT been opened.")


progress.json


stage               audit_complete
candidate                     None
accepted_members                 2
Name: Last durable stage, dtype: object

roster.json
Accepted members: ['selected_47_splines_geo_reference', 'g000_c001']
Processed candidate count: 2
summary.json


selected_47_splines_geo_reference    0.9
g000_c001                            0.1
dtype: float64

Selection: {'homes': 500, 'mdape': 11.068321311888534, 'mape': 13.911057315580218, 'p90_ape': 30.380027612683197}
Shadow audit: {'homes': 500, 'mdape': 11.274480939851024, 'mape': 14.873416372524241, 'p90_ape': 32.00591804828144}
events.json


,generation,candidate,params,tuning_score,screen,admitted,weight,marginal_gain,solo,error_corr_with_ensemble,updated_metrics,elapsed_seconds
0,0,"{'name': 'selected_47_splines_geo_reference', ...","{'n_estimators': 700, 'learning_rate': 0.04500...",11.765514,"{'homes': 250, 'mdape': 10.594628057644677, 'm...",True,1.0,NaN,"{'homes': 500, 'mdape': 11.14616466396933, 'ma...",NaN,"{'homes': 500, 'mdape': 11.14616466396933, 'ma...",33.785465
1,0,"{'name': 'g000_c001', 'seed': 942484272, 'text...","{'n_estimators': 700, 'learning_rate': 0.11480...",12.577902,"{'homes': 250, 'mdape': 10.385089034536826, 'm...",True,0.1,0.078412,"{'homes': 500, 'mdape': 11.686882044065246, 'm...",0.890462,"{'homes': 500, 'mdape': 11.068321311888534, 'm...",49.453188
